using age_depth_harmonica

In [ ]:
import matplotlib.pyplot as plt
import xarray as xr
import numpy as np
import pandas as pd
import pygmt

In [ ]:
df = pd.read_csv('..\\..\\data\\casie21\\TOC_casie21.dat', sep='\\s+', skiprows=1, 
                 names=['long', 'lat', 'toc_z'])
print(df.head())


# tesseroid time!
Now we are going to plot the gravity as a tesseroid (6 bounds: west, east, south, north, bottom, top)

In [ ]:
import boule as bl
import verde as vd
import harmonica as hm
from numba import njit


GRS80 (Geodetic Reference System 1980) vs WGS84 (World Geodetic System 1984) --> '...if you're using GPS coordinates or anything referenced to satellite positioning, you're technically in WGS84; if you're doing pure geodetic/gravity theory work, GRS80 is the traditional standard.'

In [ ]:
ellipsoid = bl.WGS84


In [ ]:
ellipsoid = bl.WGS84

a = ellipsoid.semimajor_axis
b = ellipsoid.semiminor_axis
mean_radius = (2 * a + b) / 3

### using hm.tesseroid_gravity: forward model that computes unknown gravity from a physical model

[vd.grid_coords](https://www.fatiando.org/verde/latest/api/generated/verde.grid_coordinates.html#verde-grid-coordinates) 

[hm.tesseroid_gravity](https://www.fatiando.org/harmonica/latest/api/generated/harmonica.tesseroid_gravity.html) --> from hm tesseroid example but takes too long for all the data

field options:
- grav potential: potential
- downward acc: g_z

extra_coords: adds a scalar constant array (such as a fixed height). can also make it a list for height, time, etc.

In [ ]:
bottom = ellipsoid.mean_radius - 10e3
top = ellipsoid.mean_radius

print(df['lon'].min(), df['lon'].max(), df['lat'].min(), df['lat'].max())

region=df['lon'].min(), df['lon'].max(), df['lat'].min(), df['lat'].max()
spacing=0.01  # number of pnts in each S-N and W-E direction, adjust based on data density

In [ ]:
@njit
def density(radius):
    """Linear density function"""
    top = mean_radius
    bottom = mean_radius - 10e3
    density_top = 2670
    density_bottom = 3000
    slope = (density_top - density_bottom) / (top - bottom)
    return slope * (radius - bottom) + density_bottom

In [ ]:
# define the tesseroid as 6 parameters (hence 6 bounds meow)
tesseroid = ([df['lon'].min() + 5, df['lon'].max() - 4, df['lat'].min() + 4, df['lat'].max() - 6, 
                bottom, top],
            [df['lon'].min() + 4, df['lon'].max() - 3, df['lat'].min() + 5, df['lat'].max() - 3, 
                bottom, top],
            [df['lon'].min() + 2, df['lon'].max() - 6, df['lat'].min() + 1, df['lat'].max() - 10, 
                bottom, top])


# density = 3000  # [kg/m^3] for oceanic crust
# density = np.full(tesseroid.shape[0], density)

coords = vd.grid_coordinates(
    region=region,
    spacing=spacing,
    extra_coords=mean_radius)

gravity = hm.tesseroid_gravity(coords, tesseroid, density, field='g_z')

In [ ]:
grid = vd.make_xarray_grid(coords, gravity, data_names='gravity', extra_coords_names='extra')

In [ ]:
# df['gravity'] = gravity
# df['residual'] = df['FA'] - df['gravity']

# print(df['gravity'].head())
# print(df['residual'].head())

In [ ]:
fig = pygmt.Figure()
title = 'Fwd Model - Downward Comp of Grav Acceleration'
with pygmt.config(FONT_TITLE='22p'):
   fig.grdimage(
      region=region,
      projection='M15c',
      grid=grid.gravity,
      frame=['a', f'+t{title}'],
      cmap='viridis',
   )

fig.plot(data='..\\..\\data\\Plate_Boundaries.shp\\Plate_Boundaries.shp', pen='0.5p,red')
fig.coast(land='green', transparency=50, shorelines='0.5p,darkgreen')

fig.colorbar(cmap=True, frame=['a600f100', 'x+lDensity'])

fig.savefig('..\\..\\figures\\jdf_grav\\fwd_model_0.01.png', dpi=400, crop='0.5c')  # crop to put whitespace around when i save it
fig.show()

## making a depth tesseroid now...

In [ ]:
block_reduce = pygmt.blockmedian(x=df['lon'], y=df['lat'], z=dt_ps,   # or whatever column you want gridded
    region=region,
    spacing=spacing)

# interpolate onto a continuous grid
ps_depth_grid = pygmt.surface(data=block_reduce, region=region, spacing=spacing)

print(ps_depth_grid.head())

block_reduce = pygmt.blockmedian(x=df['lon'], y=df['lat'], z=dt_stein,   # or whatever column you want gridded
    region=region,
    spacing=spacing)

# interpolate onto a continuous grid
stein_depth_grid = pygmt.surface(data=block_reduce, region=region, spacing=spacing)

print(stein_depth_grid.head())

makecpt error: if you don't set the series it may not show the data

In [ ]:
fig2 = pygmt.Figure()
title = 'Pars-Scl Estimated Seafloor Depth'

pygmt.makecpt(cmap='bathy', series=[ps_depth_grid.min().values, ps_depth_grid.max().values])

fig2.grdimage(grid=ps_depth_grid, cmap=True, region=region, projection='M15c', frame=['a', f'+t{title}'])
fig2.colorbar(cmap=True, frame=['a300f100', 'x+lm'])  # annotate every x, tick every y
fig2.plot(data='..\\..\\data\\Plate_Boundaries.shp\\Plate_Boundaries.shp', pen='0.5p,red')
fig2.coast(land='green', transparency=50, shorelines='0.5p,darkgreen')

fig2.savefig('..\\..\\figures\\jdf_grav\\depth_ps_0.01.png', dpi=400, crop='0.5c')  # crop to put whitespace around when i save it
fig2.show()

In [ ]:
fig3 = pygmt.Figure()
title = 'Stein x2 Estimated Seafloor Depth'

pygmt.makecpt(cmap='bathy', series=[stein_depth_grid.min().values, stein_depth_grid.max().values])

fig3.grdimage(grid=stein_depth_grid, cmap=True, region=region, projection='M15c', frame=['a', f'+t{title}'])
fig3.colorbar(cmap=True, frame=['a300f100', 'x+lm'])  # annotate every x, tick every y
fig3.plot(data='..\\..\\data\\Plate_Boundaries.shp\\Plate_Boundaries.shp', pen='0.5p,red')
fig3.coast(land='green', transparency=50, shorelines='0.5p,darkgreen')

fig3.savefig('..\\..\\figures\\jdf_grav\\depth_stein_0.01.png', dpi=400, crop='0.5c')  # crop to put whitespace around when i save it
fig3.show()

In [ ]:
diff_depth = stein_depth_grid - ps_depth_grid

# print(diff_depth.head())

In [ ]:
fig4 = pygmt.Figure()
title = 'Seafloor Depth Difference'

pygmt.makecpt(cmap='dense', series=[diff_depth.min().values, diff_depth.max().values])

fig4.grdimage(grid=diff_depth, cmap=True, region=region, projection='M15c', frame=['a', f'+t{title}'])
fig4.colorbar(cmap=True, frame=['a10f5', 'x+lm'])  # annotate every x, tick every y
fig4.plot(data='..\\..\\data\\Plate_Boundaries.shp\\Plate_Boundaries.shp', pen='0.75p,red')
fig4.coast(land='green', transparency=50, shorelines='0.5p,darkgreen')

fig4.savefig('..\\..\\figures\\jdf_grav\\depth_diff_0.01.png', dpi=400, crop='0.5c')  # crop to put whitespace around when i save it
fig4.show()

## plotting gebco data

In [ ]:
fig5 = pygmt.Figure()
title = 'Seabed Data'

pygmt.makecpt(cmap='abyss', series=[-5000, 1000])

fig5.grdimage(grid='..\\..\\data\\GEBCO_20_Jul_2026_602a71007e75\\seabed.grd', cmap=True, 
              region=region, projection='M15c', 
              frame=['xa2f2g1','ya2f2g1', f'+t{title}'] # annot every 2, 2 ticks inbwtn, 1 gridline
              )
fig5.colorbar(cmap=True, frame=['a1000f500', 'x+lm'])  # annotate every x, tick every y
fig5.plot(data='..\\..\\data\\Plate_Boundaries.shp\\Plate_Boundaries.shp', pen='1p,red')
fig5.coast(land='green', transparency=50, shorelines='0.5p,darkgreen')

fig5.savefig('..\\..\\figures\\jdf_grav\\seabed_grded_0.01.png', dpi=500, crop='0.5c')  # crop to put whitespace around when i save it
fig5.show()